In [9]:
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [11]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }
    
    if system:
        params["system"] = system

    message= client.messages.create(**params)
    return message.content[0].text

In [22]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON or Regex specfically for AWS-related tasks. Generate an array of JSON objects, each representing a task that requires Python, JSON or Regex to complete. 

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* focus on tasks that can be solved by writing a single Python function, a single JSON object or a single regex
* Focus on tasks that do not required writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [23]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [24]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [12]:
import json

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages= [] 
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [31]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [32]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [33]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7


In [34]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket ARN Region Extractor\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_region_from_s3_arn(arn: str) -> Optional[str]:\n    \"\"\"\n    Extract the AWS region from an S3 bucket ARN.\n    \n    S3 bucket ARNs have the format: arn:aws:s3:::bucket-name\n    S3 object ARNs have the format: arn:aws:s3:::bucket-name/key\n    \n    Note: S3 buckets themselves don't have regions in their ARN,\n    but this function can handle extended ARN formats.\n    \n    Args:\n        arn (str): The S3 ARN to parse\n        \n    Returns:\n        Optional[str]: The region string, or None if not present/applicable\n        \n    Examples:\n        >>> extract_region_from_s3_arn('arn:aws:s3:::my-bucket')\n        None\n        >>> extract_region_from_s3_arn('arn:aws:s3:us-east-1:123456789012:bucket/my-bucket')\n        'us-east-1'\n    \"\"\"\n    if not isinstance(arn, str) or not arn.strip():\n        return None\n    \n    # Standard S3 bucket ARN fo